# Agriculture Data Cleaning: USDA-NASS Livestock Inventory

Cleans the USDA-NASS Quick Stats **county-level livestock inventory** export for
Iowa into a tidy table: one row per
`(program, year, period, county, commodity_detail, statistic, herd_size_band)`.

**Input:**  `data/tabular/01_raw/agriculture/Livestock-Inventory.csv`
**Output:** `data/tabular/02_clean/agriculture/livestock-inventory-clean.csv`

**Two layers of data live in this one file** — and conflating them would
double-count animals:
- **`domain = TOTAL`**: the headline numbers — head `INVENTORY` and the count of
  `OPERATIONS WITH INVENTORY`, by species/class (cattle, hogs, goats, sheep).
- **`domain = INVENTORY OF ...`**: the *same operations* re-counted into
  **herd-size bands** (e.g. `1 TO 9 HEAD`, `500 OR MORE HEAD`). These are a
  breakdown of the operation counts, **not** additional animals.

We keep both but make the distinction explicit via a parsed `herd_size_band`
column (`NaN` for the `TOTAL` rows), so a modeler can trivially filter to the
headline series with `df.domain == "TOTAL"`.

**Pipeline**
1. **Load** the raw export as strings.
2. **Drop** empty / single-value columns (keeping the two real `Period` values).
3. **Parse** `year`, `county_fips`, `value`/`cv_pct` (suppression -> `NaN`).
4. **Unpack** `Data Item` into `commodity_detail` / `statistic`, and assign a
   meaningful `unit` (`HEAD` vs `OPERATIONS`).
5. **Parse** the `Domain Category` herd-size band.
6. **Enforce** the unique key, **sanity check**, and **save**.

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "agriculture"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "agriculture"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/agriculture
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/agriculture


In [2]:
# --- USDA-NASS shared cleaning helpers ------------------------------------
#
# NASS uses parenthetical letter codes in place of numbers. They are NOT data;
# they encode *why* a number is absent, so they must become NaN (never 0):
#   (D) withheld to avoid disclosing data for individual operations
#   (Z) value rounds to less than half the unit shown
#   (X) not applicable
#   (NA) not available
#   (H) sampling CV >= 99.95% (estimate too unreliable to publish)
#   (L) sampling CV  <  0.05%
# (D) appears in `Value`; (D)/(H)/(L) appear in `CV (%)`.
NASS_SUPPRESSION = {"(D)", "(Z)", "(X)", "(NA)", "(H)", "(L)", "(NA)", "(S)"}


def parse_nass_numeric(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    """Parse a NASS Value/CV column into (float, was_suppressed_flag).

    Strips thousands separators, maps every suppression code to NaN, and flags
    which rows were a real suppression code (vs. genuinely blank) so downstream
    users can tell "censored" apart from "not collected".
    """
    s = series.astype("string").str.strip()
    suppressed = s.isin(NASS_SUPPRESSION)
    cleaned = s.mask(s.isin(NASS_SUPPRESSION))           # codes -> <NA>
    cleaned = cleaned.str.replace(",", "", regex=False)  # 1,234 -> 1234
    return pd.to_numeric(cleaned, errors="coerce"), suppressed.fillna(False)


# A Data Item is "<COMMODITY DETAIL> - <STATISTIC>[, MEASURED IN <UNIT>]",
# e.g. "CORN, GRAIN - YIELD, MEASURED IN BU / ACRE" or "CORN - ACRES PLANTED".
_DATA_ITEM_RE = re.compile(r"^(?P<detail>.+?) - (?P<stat>.+?)(?:, MEASURED IN (?P<unit>.+))?$")


def parse_data_item(item: str) -> tuple[str, str, str | None]:
    """Split a Data Item string into (commodity_detail, statistic, unit)."""
    m = _DATA_ITEM_RE.match(item)
    if not m:
        raise ValueError(f"Unparseable Data Item: {item!r}")
    return m.group("detail"), m.group("stat"), m.group("unit")

## Step 1 - Load

In [3]:
RAW_FILE = "Livestock-Inventory.csv"
raw = pd.read_csv(RAW_DIR / RAW_FILE, dtype="string")
n_raw = len(raw)
print(f"Loaded {n_raw:,} rows x {raw.shape[1]} cols from {RAW_FILE}")
raw.head()

Loaded 21,868 rows x 21 cols from Livestock-Inventory.csv


,Program,Year,Period,Week Ending,Geo Level,State,State ANSI,Ag District,Ag District Code,County,...,Zip Code,Region,watershed_code,Watershed,Commodity,Data Item,Domain,Domain Category,Value,CV (%)
0,CENSUS,2022,END OF DEC,<NA>,COUNTY,IOWA,19,CENTRAL,50,BOONE,...,<NA>,<NA>,00000000,<NA>,CATTLE,"CATTLE, (EXCL COWS) - INVENTORY","INVENTORY OF CATTLE, (EXCL COWS)","INVENTORY OF CATTLE, (EXCL COWS): (1 TO 9 HEAD)",275,13.4
1,CENSUS,2022,END OF DEC,<NA>,COUNTY,IOWA,19,CENTRAL,50,BOONE,...,<NA>,<NA>,00000000,<NA>,CATTLE,"CATTLE, (EXCL COWS) - INVENTORY","INVENTORY OF CATTLE, (EXCL COWS)","INVENTORY OF CATTLE, (EXCL COWS): (10 TO 19 HEAD)",275,13.2
2,CENSUS,2022,END OF DEC,<NA>,COUNTY,IOWA,19,CENTRAL,50,BOONE,...,<NA>,<NA>,00000000,<NA>,CATTLE,"CATTLE, (EXCL COWS) - INVENTORY","INVENTORY OF CATTLE, (EXCL COWS)","INVENTORY OF CATTLE, (EXCL COWS): (100 TO 199 ...",922,12.6
3,CENSUS,2022,END OF DEC,<NA>,COUNTY,IOWA,19,CENTRAL,50,BOONE,...,<NA>,<NA>,00000000,<NA>,CATTLE,"CATTLE, (EXCL COWS) - INVENTORY","INVENTORY OF CATTLE, (EXCL COWS)","INVENTORY OF CATTLE, (EXCL COWS): (20 TO 49 HEAD)",460,12.6
4,CENSUS,2022,END OF DEC,<NA>,COUNTY,IOWA,19,CENTRAL,50,BOONE,...,<NA>,<NA>,00000000,<NA>,CATTLE,"CATTLE, (EXCL COWS) - INVENTORY","INVENTORY OF CATTLE, (EXCL COWS)","INVENTORY OF CATTLE, (EXCL COWS): (200 TO 499 ...","1,987",15.1


In [4]:
# Every Data Item must parse, and every State ANSI must be Iowa (19) -- guard
# against a future re-pull silently changing the schema or scope.
assert raw["State ANSI"].dropna().eq("19").all(), "Non-Iowa rows present!"
_unparsed = [it for it in raw["Data Item"].dropna().unique() if not _DATA_ITEM_RE.match(it)]
assert not _unparsed, f"Unparseable Data Items: {_unparsed}"
print("Schema guards passed: all Iowa, all Data Items parse.")

Schema guards passed: all Iowa, all Data Items parse.


## Step 2 - Drop structurally-empty and constant columns

Same idea as the crop notebook, but here **`Period` is kept** — it carries two
real values (`FIRST OF JAN` survey vs `END OF DEC` census reference points), and
**`Domain` / `Domain Category` are kept** because they encode the herd-size
breakdown described above.

In [5]:
EMPTY_COLS = ["Week Ending", "Zip Code", "Region", "Watershed"]
CONST_COLS = ["Geo Level", "State", "watershed_code"]

for col in EMPTY_COLS:
    assert raw[col].isna().all(), f"Expected {col!r} empty but it has values"
for col in CONST_COLS:
    assert raw[col].nunique(dropna=True) <= 1, f"Expected {col!r} constant: {raw[col].unique()}"

df = raw.drop(columns=EMPTY_COLS + CONST_COLS)
print(f"Dropped {len(EMPTY_COLS)} empty + {len(CONST_COLS)} constant columns; "
      f"{df.shape[1]} columns remain")
print("Periods kept:", df["Period"].unique().tolist())

Dropped 4 empty + 3 constant columns; 14 columns remain
Periods kept: ['END OF DEC', 'FIRST OF JAN']


## Step 3 - Parse keys & values

Identical mechanics to the crop notebook: integer `year`, 5-digit `county_fips`
(blank for the `OTHER COUNTIES` rollups), and NASS-aware numeric parsing of
`Value` and `CV (%)` with a `(D)` suppression flag.

In [6]:
df["year"] = df["Year"].astype(int)

state = df["State ANSI"].str.zfill(2)
county = df["County ANSI"].str.zfill(3)
df["county_fips"] = (state + county).where(df["County ANSI"].notna())

df["value"], df["value_suppressed"] = parse_nass_numeric(df["Value"])
df["cv_pct"], _ = parse_nass_numeric(df["CV (%)"])

print(f"value: {df['value'].notna().sum():,} numeric, "
      f"{df['value_suppressed'].sum():,} suppressed (D) "
      f"({df['value_suppressed'].mean():.1%})")

value: 19,166 numeric, 2,702 suppressed (D) (12.4%)


## Step 4 - Unpack `Data Item` and assign a real `unit`

Livestock `Data Item`s have no `MEASURED IN` clause, so the raw unit is blank.
But the statistic fully determines it: an `INVENTORY` is a count of **head**, and
`OPERATIONS WITH INVENTORY` is a count of **operations** (farms). We fill `unit`
accordingly so the cleaned table is self-documenting.

In [7]:
parsed = df["Data Item"].map(parse_data_item)
df["commodity_detail"] = parsed.map(lambda t: t[0])
df["statistic"] = parsed.map(lambda t: t[1])

UNIT_BY_STAT = {"INVENTORY": "HEAD", "OPERATIONS WITH INVENTORY": "OPERATIONS"}
assert set(df["statistic"].unique()) <= set(UNIT_BY_STAT), \
    f"Unexpected statistic: {set(df['statistic'].unique()) - set(UNIT_BY_STAT)}"
df["unit"] = df["statistic"].map(UNIT_BY_STAT)
print(df.groupby(["statistic", "unit"]).size().to_string())

statistic                  unit      
INVENTORY                  HEAD          12481
OPERATIONS WITH INVENTORY  OPERATIONS     9387


## Step 5 - Parse the herd-size band from `Domain Category`

`Domain Category` is either `NOT SPECIFIED` (the `TOTAL` headline rows) or
`INVENTORY OF <class>: (<band> HEAD)`. We extract just the band
(e.g. `1 TO 9`, `500 OR MORE`) into `herd_size_band`, leaving `NaN` for the
headline rows.

In [8]:
# Pull the text inside the parentheses, then strip a trailing " HEAD".
band = df["Domain Category"].str.extract(r":\s*\((?P<band>.+?)\)$")["band"]
df["herd_size_band"] = band.str.replace(r"\s*HEAD$", "", regex=True)

df = df.rename(columns={
    "Program": "program",
    "Period": "period",
    "Commodity": "commodity",
    "County": "county",
    "Ag District": "ag_district",
    "Ag District Code": "ag_district_code",
    "Domain": "domain",
})

# Cross-check: a band is present iff this is a breakdown row (domain != TOTAL).
is_breakdown = df["domain"].ne("TOTAL")
assert (df["herd_size_band"].notna() == is_breakdown).all(), "band/domain mismatch"
print(f"Headline (TOTAL) rows: {(~is_breakdown).sum():,}  |  "
      f"herd-size-band rows: {is_breakdown.sum():,}")
print("Bands:", sorted(df["herd_size_band"].dropna().unique(),
                       key=lambda b: (len(b), b)))

Headline (TOTAL) rows: 7,646  |  herd-size-band rows: 14,222
Bands: ['1 TO 9', '1 TO 19', '1 TO 24', '10 TO 19', '20 TO 49', '25 TO 49', '25 TO 99', '50 TO 99', '100 TO 199', '100 TO 299', '200 TO 499', '300 TO 999', '500 TO 999', '500 OR MORE', '1,000 OR MORE']


## Step 6 - Select, order, and enforce a unique key

In [9]:
OUTPUT_COLS = [
    "program", "year", "period",
    "ag_district_code", "ag_district", "county", "county_fips",
    "commodity", "commodity_detail", "statistic", "unit",
    "domain", "herd_size_band",
    "value", "value_suppressed", "cv_pct",
]
clean = df[OUTPUT_COLS].sort_values(
    ["program", "year", "period", "ag_district_code", "county",
     "commodity_detail", "statistic", "herd_size_band"]
).reset_index(drop=True)

KEY = ["program", "year", "period", "ag_district_code", "county",
       "commodity_detail", "statistic", "domain", "herd_size_band"]
dupes = clean.duplicated(KEY).sum()
assert dupes == 0, f"{dupes} duplicate key rows!"
print(f"Key is unique across {len(clean):,} rows.")
clean.head()

Key is unique across 21,868 rows.


,program,year,period,ag_district_code,ag_district,county,county_fips,commodity,commodity_detail,statistic,unit,domain,herd_size_band,value,value_suppressed,cv_pct
0,CENSUS,2017,END OF DEC,10,NORTHWEST,BUENA VISTA,19021,CATTLE,"CATTLE, (EXCL COWS)",INVENTORY,HEAD,"INVENTORY OF CATTLE, (EXCL COWS)",1 TO 9,73,False,49.0
1,CENSUS,2017,END OF DEC,10,NORTHWEST,BUENA VISTA,19021,CATTLE,"CATTLE, (EXCL COWS)",INVENTORY,HEAD,"INVENTORY OF CATTLE, (EXCL COWS)",10 TO 19,278,False,31.7
2,CENSUS,2017,END OF DEC,10,NORTHWEST,BUENA VISTA,19021,CATTLE,"CATTLE, (EXCL COWS)",INVENTORY,HEAD,"INVENTORY OF CATTLE, (EXCL COWS)",100 TO 199,1696,False,55.8
3,CENSUS,2017,END OF DEC,10,NORTHWEST,BUENA VISTA,19021,CATTLE,"CATTLE, (EXCL COWS)",INVENTORY,HEAD,"INVENTORY OF CATTLE, (EXCL COWS)",20 TO 49,758,False,33.4
4,CENSUS,2017,END OF DEC,10,NORTHWEST,BUENA VISTA,19021,CATTLE,"CATTLE, (EXCL COWS)",INVENTORY,HEAD,"INVENTORY OF CATTLE, (EXCL COWS)",200 TO 499,3960,False,44.5


## Step 7 - Sanity check

In [10]:
print(f"Rows: {len(clean):,}  |  years: {clean['year'].min()}-{clean['year'].max()}  |  "
      f"counties (with FIPS): {clean['county_fips'].nunique():,}")
print(f"programs: {clean['program'].value_counts().to_dict()}")
print(f"periods:  {clean['period'].value_counts().to_dict()}\n")
print("Headline total head inventory by species (domain==TOTAL, statistic==INVENTORY):")
head = clean[(clean.domain == "TOTAL") & (clean.statistic == "INVENTORY") & clean.value.notna()]
print(head.groupby("commodity")["value"].agg(["count", "min", "median", "max"]).to_string())

Rows: 21,868  |  years: 2015-2025  |  counties (with FIPS): 99
programs: {'CENSUS': 18774, 'SURVEY': 3094}
periods:  {'END OF DEC': 18774, 'FIRST OF JAN': 3094}

Headline total head inventory by species (domain==TOTAL, statistic==INVENTORY):
           count  min    median      max
commodity                               
CATTLE      3406    5   13000.0   425000
GOATS        480    3     324.5    11726
HOGS         192  119  145541.0  1486642
SHEEP        247   16     965.0    18334


## Step 8 - Save

In [11]:
out_file = CLEAN_DIR / "livestock-inventory-clean.csv"
clean.to_csv(out_file, index=False)
print(f"Saved {len(clean):,} rows -> {out_file}")

Saved 21,868 rows -> /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/agriculture/livestock-inventory-clean.csv
